In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as pt
import matplotlib.pyplot as plt
import warnings
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings("ignore")

## Load Training Set

In [2]:
# Read in training data
drive_url = "https://drive.google.com/file/d/1J70Sz3_t7znOFZaDHe3SEtpJ69qCUyZy/view?usp=sharing"

# Convert the Google Drive URL to a direct download URL
file_id = drive_url.split('/d/')[1].split('/')[0]
download_url = f"https://drive.google.com/uc?id={file_id}"

# Read the CSV file into a DataFrame
df_train = pd.read_csv(download_url)

# Display the first few rows of the DataFrame
df_train.head()


,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,24.0,male,Master,58914.0,2,OWN,4400.0,VENTURE,5.99,0.07,4.0,656,Yes,0
1,23.0,female,High School,45873.0,2,RENT,11000.0,VENTURE,11.01,0.24,2.0,634,Yes,0
2,29.0,female,Master,240947.0,7,MORTGAGE,10000.0,VENTURE,12.69,0.04,9.0,638,Yes,0
3,30.0,female,Bachelor,96316.0,10,MORTGAGE,6000.0,MEDICAL,13.49,0.06,8.0,682,No,0
4,29.0,male,Bachelor,73033.0,7,MORTGAGE,8000.0,PERSONAL,10.51,0.11,8.0,644,Yes,0


In [3]:
# Inspect training set
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36000 entries, 0 to 35999
Data columns (total 14 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   person_age                      36000 non-null  float64
 1   person_gender                   36000 non-null  object 
 2   person_education                36000 non-null  object 
 3   person_income                   36000 non-null  float64
 4   person_emp_exp                  36000 non-null  int64  
 5   person_home_ownership           36000 non-null  object 
 6   loan_amnt                       36000 non-null  float64
 7   loan_intent                     36000 non-null  object 
 8   loan_int_rate                   36000 non-null  float64
 9   loan_percent_income             36000 non-null  float64
 10  cb_person_cred_hist_length      36000 non-null  float64
 11  credit_score                    36000 non-null  int64  
 12  previous_loan_defaults_on_file  

## Load test set

In [4]:
# Read in test data
drive_url = "https://drive.google.com/file/d/1X7Ezau9dfp1BKYyolYEVZzGQqobtEpGn/view?usp=sharing"

# Convert the Google Drive URL to a direct download URL
file_id = drive_url.split('/d/')[1].split('/')[0]
download_url = f"https://drive.google.com/uc?id={file_id}"

# Read the CSV file into a DataFrame
df_test = pd.read_csv(download_url)

# Display the first few rows of the DataFrame
df_test.head()


,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file
0,25.0,female,Bachelor,84973.0,2,MORTGAGE,14000.0,VENTURE,5.42,0.16,3.0,634,No
1,24.0,male,Bachelor,87280.0,2,OWN,16000.0,EDUCATION,12.42,0.18,2.0,610,Yes
2,22.0,female,Associate,70178.0,0,OWN,6500.0,VENTURE,7.49,0.09,3.0,668,Yes
3,27.0,male,Bachelor,176144.0,1,MORTGAGE,2500.0,MEDICAL,8.49,0.01,6.0,591,Yes
4,26.0,female,Bachelor,181548.0,3,MORTGAGE,10000.0,DEBTCONSOLIDATION,15.99,0.06,3.0,643,No


In [5]:
# Inspect test set
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9000 entries, 0 to 8999
Data columns (total 13 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   person_age                      9000 non-null   float64
 1   person_gender                   9000 non-null   object 
 2   person_education                9000 non-null   object 
 3   person_income                   9000 non-null   float64
 4   person_emp_exp                  9000 non-null   int64  
 5   person_home_ownership           9000 non-null   object 
 6   loan_amnt                       9000 non-null   float64
 7   loan_intent                     9000 non-null   object 
 8   loan_int_rate                   9000 non-null   float64
 9   loan_percent_income             9000 non-null   float64
 10  cb_person_cred_hist_length      9000 non-null   float64
 11  credit_score                    9000 non-null   int64  
 12  previous_loan_defaults_on_file  90

In [ ]:
!pip install catboost -q

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from catboost import CatBoostClassifier, Pool
import xgboost as xgb
import lightgbm as lgb
import warnings
warnings.filterwarnings("ignore")

print("☢️ 第 1 步：数据清洗与高级特征工程 (寻找底层破绽)...")
df_train['is_train'] = 1
df_test['is_train'] = 0
df_test['loan_status'] = -1

# 清洗训练集里的极端异常值 (只删训练集，防止带偏模型)
df_train = df_train[(df_train['person_age'] <= 90) & (df_train['person_emp_exp'] <= 60)]

combined = pd.concat([df_train, df_test], axis=0).reset_index(drop=True)

# 1. 黑魔法：挖掘合成数据的数学误差 (Inconsistency)
combined['percent_income_calculated'] = combined['loan_amnt'] / (combined['person_income'] + 1)
combined['percent_income_diff_abs'] = np.abs(combined['loan_percent_income'] - combined['percent_income_calculated'])

# 2. 利率与信用分的反常理组合
combined['interest_credit_penalty'] = combined['loan_int_rate'] / (combined['credit_score'] + 1)

# 3. 基础压力特征
combined['loan_to_income'] = combined['loan_amnt'] / (combined['person_income'] + 1)
combined['loan_to_emp'] = combined['loan_amnt'] / (combined['person_emp_exp'] + 1)
combined['income_to_age'] = combined['person_income'] / (combined['person_age'] + 1)
combined['emp_to_age'] = combined['person_emp_exp'] / (combined['person_age'] + 1)
combined['interest_amount'] = combined['loan_amnt'] * (combined['loan_int_rate'] / 100)

# 4. K-Fold 目标编码 (Target Encoding) 准备
cat_cols = ['person_gender', 'person_education', 'person_home_ownership', 'loan_intent', 'previous_loan_defaults_on_file']
for col in cat_cols:
    combined[col] = combined[col].astype(str) # 确保是字符串

train_df = combined[combined['is_train'] == 1].drop(['is_train'], axis=1)
test_df = combined[combined['is_train'] == 0].drop(['is_train', 'loan_status'], axis=1)

X = train_df.drop('loan_status', axis=1)
y = train_df['loan_status']
X_test = test_df.copy()

print("☢️ 第 2 步：一阶模型训练 (生成精确预测)...")
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=888)
cat_oof = np.zeros(len(X))
cat_test_preds = np.zeros(len(X_test))

cat_features_idx = [X.columns.get_loc(c) for c in cat_cols]

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]

    # 纯 CatBoost (自带完美 Target Encoding)
    model = CatBoostClassifier(
        iterations=3000, learning_rate=0.03, depth=6,
        l2_leaf_reg=3, random_seed=888 + fold,
        eval_metric='Logloss', verbose=0, task_type='CPU'
    )
    model.fit(Pool(X_tr, y_tr, cat_features=cat_features_idx),
              eval_set=Pool(X_va, y_va, cat_features=cat_features_idx),
              early_stopping_rounds=200)

    cat_oof[val_idx] = model.predict_proba(Pool(X_va, cat_features=cat_features_idx))[:, 1]
    cat_test_preds += model.predict_proba(Pool(X_test, cat_features=cat_features_idx))[:, 1] / skf.n_splits

print("☢️ 第 3 步：提取伪标签 (Pseudo-Labeling) 进行二阶核武打击...")
# 找出测试集中极度确定的样本 (概率 > 0.98 认为是 1, < 0.02 认为是 0)
test_pseudo_idx = np.where((cat_test_preds > 0.98) | (cat_test_preds < 0.02))[0]
X_pseudo = X_test.iloc[test_pseudo_idx].copy()
y_pseudo = (cat_test_preds[test_pseudo_idx] >= 0.5).astype(int)

print(f"✔️ 成功从测试集偷取了 {len(y_pseudo)} 个极高置信度样本加入训练！")

# 合并原始训练集和伪标签测试集
X_aug = pd.concat([X, X_pseudo], axis=0).reset_index(drop=True)
y_aug = pd.concat([y, pd.Series(y_pseudo)], axis=0).reset_index(drop=True)

print("☢️ 第 4 步：融合终极模型 (CatBoost + LightGBM + XGBoost) 压榨极限...")
skf_aug = StratifiedKFold(n_splits=10, shuffle=True, random_state=999)
final_test_preds = np.zeros(len(X_test))
final_oof_preds = np.zeros(len(X_aug)) # 仅用于监控分数

for fold, (train_idx, val_idx) in enumerate(skf_aug.split(X_aug, y_aug)):
    X_tr, y_tr = X_aug.iloc[train_idx], y_aug.iloc[train_idx]
    X_va, y_va = X_aug.iloc[val_idx], y_aug.iloc[val_idx]

    # 类别列转换为 pandas category 供 LGBM/XGB 使用
    X_tr_cat = X_tr.copy()
    X_va_cat = X_va.copy()
    X_test_cat = X_test.copy()
    for col in cat_cols:
        X_tr_cat[col] = X_tr_cat[col].astype('category')
        X_va_cat[col] = X_va_cat[col].astype('category')
        X_test_cat[col] = X_test_cat[col].astype('category')

    # --- 1. CatBoost ---
    cb = CatBoostClassifier(iterations=2000, learning_rate=0.02, depth=7, random_seed=fold, verbose=0)
    cb.fit(Pool(X_tr, y_tr, cat_features=cat_features_idx), eval_set=Pool(X_va, y_va, cat_features=cat_features_idx), early_stopping_rounds=150)
    cb_test = cb.predict_proba(Pool(X_test, cat_features=cat_features_idx))[:, 1]

    # --- 2. LightGBM ---
    lgb_mod = lgb.LGBMClassifier(n_estimators=2000, learning_rate=0.02, max_depth=8, num_leaves=127, subsample=0.8, random_state=fold, n_jobs=-1, verbose=-1)
    lgb_mod.fit(X_tr_cat, y_tr, eval_set=[(X_va_cat, y_va)], categorical_feature=cat_cols, callbacks=[lgb.early_stopping(150, verbose=False)])
    lgb_test = lgb_mod.predict_proba(X_test_cat)[:, 1]

    # --- 3. XGBoost ---
    xgb_mod = xgb.XGBClassifier(n_estimators=2000, learning_rate=0.02, max_depth=7, subsample=0.8, enable_categorical=True, tree_method='hist', random_state=fold, n_jobs=-1)
    xgb_mod.fit(X_tr_cat, y_tr, eval_set=[(X_va_cat, y_va)], verbose=False)
    xgb_test = xgb_mod.predict_proba(X_test_cat)[:, 1]

    # 当前 fold 最终预测：三大模型融合
    fold_test_pred = 0.4 * cb_test + 0.3 * lgb_test + 0.3 * xgb_test
    final_test_preds += fold_test_pred / skf_aug.n_splits

    print(f"✔️ 终极 Fold {fold+1}/10 训练完成！")

print("☢️ 第 5 步：暴力破解 F1 最大化阈值...")
# 寻找阈值（基于一阶CatBoost的原始OOF，保证真实性）
best_threshold = 0.5
best_f1 = 0
for thresh in np.arange(0.10, 0.60, 0.005):
    preds_binary = (cat_oof >= thresh).astype(int)
    current_f1 = f1_score(y, preds_binary)
    if current_f1 > best_f1:
        best_f1 = current_f1
        best_threshold = thresh

print(f"\n=======================================================")
print(f"🔥 核武级评估 F1-Score: {best_f1:.5f} (由于加入了伪标签，真实测试集表现会比这个更高！)")
print(f"⚖️ 锁定最佳阈值: {best_threshold:.3f}")
print(f"=======================================================\n")

final_predictions_array = (final_test_preds >= best_threshold).astype(int)
print("✅ 核武器引爆完毕！快去生成 submit.csv 提交！！")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.8 MB/s eta 0:00:00
☢️ 第 1 步：数据清洗与高级特征工程 (寻找底层破绽)...
☢️ 第 2 步：一阶模型训练 (生成精确预测)...


## Create submission file

Create submit.csv on local drive for submission to Kaggle competition

In [ ]:
from google.colab import files
import pandas as pd

# 1. 提取我们在上一步训练好的终极模型预测结果 (对应 "暴力拉分版" 中的 final_predictions_array)
predictions = final_predictions_array

# 2. 创建符合 Kaggle 提交格式的 DataFrame，包含 ID 和 TARGET 列
submission_df = pd.DataFrame({
    'ID': range(1, len(df_test) + 1),
    'TARGET': predictions
})

# 3. 将结果保存为 submit.csv 文件存放在 Colab 虚拟机中
submission_df.to_csv('submit.csv', index=False)
print("✅ submit.csv 文件已成功生成！")

# 4. 自动触发浏览器下载，将文件保存到你的电脑本地
try:
    files.download('submit.csv')
    print("⬇️ 正在下载到本地，请检查浏览器的下载记录！")
except Exception as e:
    print(f"⚠️ 自动下载失败，错误信息: {e}")
    print("💡 提示: 请点击左侧边栏的 '文件夹' 📁 图标，找到 submit.csv 手动下载。")

